In [1]:
import joblib
import pandas as pd

df_users = pd.read_parquet("data\\11092026\\users.parquet")
df_movies = pd.read_parquet("data\\11092026\\movies.parquet")
df_interactions = pd.read_parquet("data\\11092026\\interactions.parquet")

encoders = joblib.load("data\\11092026\\encoders.pkl")
metadata = joblib.load("data\\11092026\\metadata.pkl")

In [2]:
df_users.head(10)

,user_id,gender,age_group,occupation,zip_code
0,1,0,1,10,48067
1,2,1,56,16,70072
2,3,1,25,15,55117
3,4,1,45,7,02460
4,5,1,25,20,55455
5,6,0,50,9,55117
6,7,1,35,1,06810
7,8,1,25,12,11413
8,9,1,25,17,61614
9,10,0,35,1,95370


In [3]:


df_movies.head(5)

,movie_id,title,genre,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),Animation|Children's|Comedy,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children's|Fantasy,0,1,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
df_interactions.head(10)

df_interactions["movie_id"] = (
    encoders["movie_encoder"].inverse_transform(df_interactions["movie_idx"])
)

df_interactions["user_id"] = (
    encoders["user_encoder"].inverse_transform(df_interactions["user_idx"])
)

df_interactions


,user_idx,movie_idx,rating,timestamp,movie_id,user_id
0,0,1104,5.0,978300760,1193,1
1,0,639,3.0,978302109,661,1
2,0,853,3.0,978301968,914,1
3,0,3177,4.0,978300275,3408,1
4,0,2162,5.0,978824291,2355,1
...,...,...,...,...,...,...
1000204,6039,1019,1.0,956716541,1091,6040
1000205,6039,1022,5.0,956704887,1094,6040
1000206,6039,548,5.0,956704746,562,6040
1000207,6039,1024,4.0,956715648,1096,6040


In [ ]:
metadata


{'n_movies': 3706, 'n_users': 6040}

In [ ]:
encoders


{'user_encoder': LabelEncoder(), 'movie_encoder': LabelEncoder()}

In [7]:
df_full.columns

NameError: name 'df_full' is not defined

In [8]:
df_full = df_users.merge(df_interactions, on="user_id", how="left")
df_full = df_full.merge(df_movies, on="movie_id", how="left")



In [60]:
df_full

,user_id,gender,age_group,occupation,zip_code,user_idx,movie_idx,rating,timestamp,movie_id,...,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,year,title_clean,rating_binary
0,1,0,1,10,48067,0,1104,5.0,978300760,1193,...,0,0,0,0,0,0,0,1975,One Flew Over the Cuckoo's Nest,1
1,1,0,1,10,48067,0,639,3.0,978302109,661,...,1,0,0,0,0,0,0,1996,James and the Giant Peach,0
2,1,0,1,10,48067,0,853,3.0,978301968,914,...,1,0,1,0,0,0,0,1964,My Fair Lady,0
3,1,0,1,10,48067,0,3177,4.0,978300275,3408,...,0,0,0,0,0,0,0,2000,Erin Brockovich,1
4,1,0,1,10,48067,0,2162,5.0,978824291,2355,...,0,0,0,0,0,0,0,1998,"Bug's Life, A",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1000204,6040,1,25,6,11106,6039,1019,1.0,956716541,1091,...,0,0,0,0,0,0,0,1989,Weekend at Bernie's,0
1000205,6040,1,25,6,11106,6039,1022,5.0,956704887,1094,...,0,0,1,0,0,1,0,1992,"Crying Game, The",1
1000206,6040,1,25,6,11106,6039,548,5.0,956704746,562,...,0,0,0,0,0,0,0,1995,Welcome to the Dollhouse,1
1000207,6040,1,25,6,11106,6039,1024,4.0,956715648,1096,...,0,0,0,0,0,0,0,1982,Sophie's Choice,1


In [ ]:
df_full.columns

drop_cols = [
    "movie_idx",
    "user_idx",
    "genre",
    "title",
    "timestamp",
    "rating",
    "rating_binary"
]
df_full.drop(columns=drop_cols)

In [9]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.temporal_split import TemporalSplit


In [ ]:

df_full["rating_binary"] = (df_full["rating"] >= 4).astype(int)
df_full["year"] = (df_full["title"].str.extract(r"\((\d+)\)")).astype(int)

drop_cols = [
    "movie_idx",
    "user_idx",
    "genre",
    "title",
    "timestamp",
    "zip_code",
    "rating",
    "rating_binary",
    "title"
]

cat_features = [
    "user_id",
    "movie_id",
    "gender",
    "age_group",
    "occupation",
]

train , val , test = TemporalSplit().split(data=df_full)

group_train = train["user_id"]
group_val = val["user_id"]

y_train = train["rating_binary"]
y_val = val["rating_binary"]
y_test = test["rating_binary"]

X_train = train.drop(columns=drop_cols)
X_val = val.drop(columns=drop_cols)
X_test = test.drop(columns=drop_cols)

In [27]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

X_train.columns

(797758, 27)
(96719, 24)
(105732, 24)


Index(['user_id', 'gender', 'age_group', 'occupation', 'movie_id', 'Action',
       'Adventure', 'Animation', 'Children's', 'Comedy', 'Crime',
       'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical',
       'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'year',
       'avg_rating', 'count_rating', 'std_rating'],
      dtype='str')

In [18]:
from catboost import CatBoostRanker

model = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=10",
    iterations=500,
    learning_rate=0.01,
    depth=5,
    random_seed=42,
    verbose=50
)

model.fit(
    X_train,
    y_train,
    group_id=group_train,
    cat_features=cat_features
)

Groupwise loss function. OneHotMaxSize set to 10
0:	total: 928ms	remaining: 7m 42s
50:	total: 35.9s	remaining: 5m 15s
100:	total: 1m 11s	remaining: 4m 42s
150:	total: 1m 47s	remaining: 4m 8s
200:	total: 2m 22s	remaining: 3m 32s
250:	total: 2m 57s	remaining: 2m 56s
300:	total: 3m 32s	remaining: 2m 20s
350:	total: 4m 7s	remaining: 1m 45s
400:	total: 4m 43s	remaining: 1m 9s
450:	total: 5m 18s	remaining: 34.6s
499:	total: 5m 52s	remaining: 0us


CatBoostRanker(depth=5, eval_metric='NDCG:top=10', iterations=500, learning_rate=0.01, loss_function='YetiRank', random_seed=42, verbose=50)

In [19]:
import joblib

model_s = {
    "model": model
}

joblib.dump(
    model_s,
    "data\\models\\catboost_old.pkl"
)

['data\\models\\catboost_old.pkl']

In [ ]:
import joblib

model = joblib.load("data\\models\\catboost_old.pkl")["model"]
model

CatBoostRanker(depth=5, eval_metric='NDCG:top=10', iterations=500, learning_rate=0.01, loss_function='YetiRank', random_seed=42, verbose=50)

In [30]:
import numpy as np
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.metrics.metrics import (
    recall_at_k,
    precision_at_k,
    ndcg_at_k,
)

movies_ids = df_movies.movie_id.unique()

train_seen = (
    train
    .groupby("user_id")["movie_id"]
    .agg(set)
    .to_dict()
)

k=10
user_id = 1

watched = train_seen.get(user_id, set())

X = pd.DataFrame({
    "user_id": user_id,
    "movie_id": movies_ids
})

X = X[
    ~X["movie_id"].isin(watched)
]

X = X.merge(
    df_users,
    on="user_id",
    how="left"
)

X = X.merge(
    df_movies,
    on="movie_id",
    how="left"
)

X = X.merge(
    df_interactions[["user_id", "rating"]],
    on="user_id",
    how="left"
)

X["year"] = (X["title"].str.extract(r"\((\d+)\)")).astype(int)

movie_ids_recomendations = X["movie_id"].copy()

drop_cols = [
    "genre",
    "zip_code",
    "title",
    "rating"
]



train_val = (df_full[
    df_full["timestamp"] <= val["timestamp"].max()
    ]
)

table = (train_val[
    train_val["user_id"] != user_id
    ]
    .groupby("movie_id")
    .agg(
        avg_rating=("rating", "mean"),
        count_rating=("rating", "count"),
        std_rating=("rating", "std")
    )
    .reset_index()
)

X = X.merge(
    table,
    on="movie_id",
    how="left"
)

features = X.drop(
    columns=drop_cols
)

print(X.columns)


print(X.head(5))

scores = model.predict(features)

top_indices = np.argsort(scores)[::-1][:k]

recomend = movie_ids_recomendations.iloc[
    top_indices
]

val_per_user = val[
    val["user_id"] == user_id
]
print(val_per_user)

relevant = val_per_user.loc[
    val["rating"] >= 4,
    "movie_id"
]

recall = recall_at_k(relevant, recomend)
precision = precision_at_k(relevant, recomend)
ndcg = ndcg_at_k(relevant, recomend)

print(recall)
print(precision)
print(ndcg)

Index(['user_id', 'movie_id', 'gender', 'age_group', 'occupation', 'zip_code',
       'title', 'genre', 'Action', 'Adventure', 'Animation', 'Children's',
       'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
       'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western', 'rating', 'year', 'avg_rating', 'count_rating',
       'std_rating'],
      dtype='str')
   user_id  movie_id  gender  age_group  occupation zip_code  \
0        1         1       0          1          10    48067   
1        1         1       0          1          10    48067   
2        1         1       0          1          10    48067   
3        1         1       0          1          10    48067   
4        1         1       0          1          10    48067   

              title                        genre  Action  Adventure  ...  \
0  Toy Story (1995)  Animation|Children's|Comedy       0          0  ...   
1  Toy Story (1995)  Animation|Children's|Comedy 

In [ ]:
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.metrics.metrics import (
    recall_at_k,
    precision_at_k,
    ndcg_at_k,
)
import numpy as np

drop_cols = [
    "genre",
    "zip_code",
    "title",
]

def eval(
        model,
        train,
        val,
        df_movie,
        df_users,
        df_interactions,
        k=10,
):

    movies_ids = df_movie.movie_id.unique()

    train_seen = (
        train
        .groupby("user_id")["movie_id"]
        .agg(set)
        .to_dict()
    )

    recalls = []
    precisions = []
    ndcgs = []


    movie_features = df_movie.copy()

    movie_features["year"] = (
        movie_features["title"]
        .str.extract(r"\((\d{4})\)")
        .astype("Int64")
    )

    movie_features = movie_features.drop(columns=["genre", "title"])

    user_features = df_users.copy()
    user_features = user_features.drop(columns="zip_code")

    for user_id in val.user_id.unique():

        watched = train_seen.get(user_id, set())

        X = pd.DataFrame({
            "user_id": user_id,
            "movie_id": movies_ids
        })

        candidates = movie_features[
            ~movie_features["movie_id"].isin(watched)
        ]

        candidates["user_id"] = user_id

        candidates = candidates.merge(user_features, on="user_id", how="left")

        candidates = candidates[X_train.columns]

        scores = model.predict(candidates)

        top_indices = np.argsort(scores)[::-1][:k]

        recommended = (
            candidates["movie_id"]
            .iloc[top_indices]
            .tolist()
        )

        val_per_user = val[
            val["user_id"] == user_id
        ]

        relevant = val_per_user.loc[
            val["rating"] >= 4,
            "movie_id"
        ]

        recall = recall_at_k(relevant, recommended)
        precision = precision_at_k(relevant, recommended)
        ndcg = ndcg_at_k(relevant, recommended)

        recalls.append(recall)
        precisions.append(precision)
        ndcgs.append(ndcg)

    return {
        "recall": np.mean(recalls),
        "precision": np.mean(precisions),
        "ndcg": np.mean(ndcgs)
    }


In [21]:
res = eval(model, train, val, df_movies, df_users, df_interactions)

Первая версмия модели с пользователями и контентными признаками показала нозкую качество модели. После добавления user_id и movie_id качество модели на validation метриках возросло, но это от части основано на запоминания пользователей и фильмов (хотя даже так вышли не самые лучшие метрики). Поэтому далее я попробую исмпользховать двухэтапную систему рекомендаций: Popularity + Matrix Factorization, после чего CatBoost будет выполнять ранжирования кандидатов. Так же будут добавленны новые признаки: Факторы MF и оценка MF. (возможно будут так же добавленны новые признаки на основе дданных с таблиц)

'recall': np.float64(0.026596517016002793), \
 'precision': np.float64(0.022367549668874175), \
 'ndcg': np.float64(0.00024488983035845813)

In [22]:
res

{'recall': np.float64(0.023557995863224122),
 'precision': np.float64(0.01867549668874172),
 'ndcg': np.float64(0.02542515150375308)}